# Data Manipulation

A photograph of a shirt reaches a model as a grid of numbers, one per
pixel, and everything else the model touches is numbers too: its parameters,
the values it computes along the way, and the gradients used to train it. A
deep learning program spends nearly all of its time computing with such
collections of numbers. How should we store them, and how can we compute
with them quickly?

Rescaling images is a typical computation. Training a model processes its
examples in small groups called *minibatches*, and a minibatch of 256
grayscale images of $28 \times 28$ pixels holds
$256 \times 28 \times 28 = 200{,}704$ numbers. Image files store each pixel
as an integer from 0 to 255, and before a model sees the images we rescale
every pixel to a real number between 0 and 1 by dividing it by 255. In plain
Python, this takes three nested loops over the images, the rows, and the
columns, and the interpreter carries out the 200,704 divisions one at a
time. In deep learning code it takes one expression, `images / 255`, which
the library applies to the whole minibatch at once in compiled code, on a
CPU or a GPU. Deep learning code computes with whole arrays in this way
throughout, rather than with loops over their elements.

An array of numbers arranged along any number of axes is called a *tensor*.
If you know NumPy [@Harris.Millman.Walt.ea.2020], much of this section
will look familiar. The tensor classes of deep learning libraries (`ndarray`
in MXNet, `Tensor` in PyTorch and TensorFlow, `jax.Array` in JAX) resemble
NumPy's `ndarray`, with two additions. They run their operations on GPUs and
other accelerators as well as on CPUs, and the libraries can compute
derivatives of computations on them automatically, as we will see later in
this chapter.

This section shows how to create tensors and inspect their shape and type,
how to rearrange, select, and change their elements, and how to compute with
whole tensors at once, including tensors whose shapes differ. It ends with
what these computations cost in memory and how to convert tensors to other
Python objects. The minibatch of images is our running example. Where a
tensor lives, in main memory or on a GPU, changes no result in this chapter;
a later chapter shows how to choose.

We import TensorFlow under its usual alias `tf`.

In [1]:
import tensorflow as tf

## Tensor Creation

### Shape

A tensor arranges its numbers along its *axes*. A vector has one axis and a
matrix has two. The minibatch of images has four: one for the image, one for
the color channel (a grayscale image has a single channel), and one each for
the rows and the columns of pixels. Axes are numbered from 0, so for the
minibatch, axis 0 indexes the images and axis 3 the columns. The number of
axes is the *order* of a tensor, and the minibatch has order 4.

The simplest way to create a tensor is `arange(n)`, which returns a vector of
the integers from 0 up to, but not including, $n$. We ask for 32-bit
floating-point numbers, the type that we compute with throughout:

In TensorFlow, this function is called `range`.

In [2]:
x = tf.range(12, dtype=tf.float32)
x

<tf.Tensor: shape=(12,), dtype=float32, numpy=
array([ 0.,  1.,  2.,  3.,  4.,  5.,  6.,  7.,  8.,  9., 10., 11.],
      dtype=float32)>

The numbers in a tensor are its *elements*, and the number of elements is
its *size*:

In TensorFlow, the function `tf.size` returns the size.

In [3]:
tf.size(x)

<tf.Tensor: shape=(), dtype=int32, numpy=12>

The *shape* of a tensor lists its length along each axis. The shape of a
vector has one entry, which equals the size:

In [4]:
x.shape

TensorShape([12])

the figure shows tensors with zero to three axes. A single
number, a *scalar*, is a tensor with no axes; its shape is empty. The size
of every tensor is the product of the entries of its shape. For the minibatch
of images, whose shape is $(256, 1, 28, 28)$, it is
$256 \times 1 \times 28 \times 28 = 200{,}704$.

![Tensors with zero, one, two, and three axes. The shape lists the length of each axis, and the size is the product of these lengths: 1, 4, 12, and 24 elements.](https://raw.githubusercontent.com/smolix/d2l-neu/notebooks/img/ndarray-rank-ladder.svg)

### Data Types

Every tensor also has a *data type*, its `dtype`, which fixes how each
element is stored. Deep learning computes mostly with 32-bit floating-point
numbers, `float32`, which take 4 bytes each and keep about 7 significant
decimal digits. Other data need other types. The pixels of an image are
8-bit unsigned integers, `uint8`, which take one byte each and hold the
integers from 0 to 255. The data type determines how much memory a tensor
needs: the minibatch of images takes 200,704 bytes as `uint8` and four times as
much, about 0.8 MB, as `float32`. Large models also compute with 16-bit
floating-point numbers, which halve the memory again at some cost in
precision (that section).

With integer arguments, `tf.range` creates a tensor of 32-bit integers,
which is why the `arange` cell above asks for `float32`. The function `tf.cast`
converts a tensor to another type, as in `tf.cast(x, tf.int32)`.

### Initial Values

A model needs tensors with initial values before training can change them.
The functions `zeros` and `ones` create a tensor of a given shape filled with
0s or with 1s; the offsets that a layer adds to its outputs, for example,
usually start at zero. Here is a tensor of zeros with the shape $(2, 3, 4)$
of the figure:

In [5]:
tf.zeros((2, 3, 4))

<tf.Tensor: shape=(2, 3, 4), dtype=float32, numpy=
array([[[0., 0., 0., 0.],
        [0., 0., 0., 0.],
        [0., 0., 0., 0.]],

       [[0., 0., 0., 0.],
        [0., 0., 0., 0.],
        [0., 0., 0., 0.]]], dtype=float32)>

The weights of a model, by contrast, start at random values. If all the
weights of a layer started at the same value, all its outputs would be
equal and would receive equal updates during training, so they could never
come to differ. Random initial values *break this symmetry*. The following
tensor holds numbers drawn independently from the standard normal
distribution, a bell-shaped distribution centered at 0 that places about 95%
of its draws between $-2$ and $2$:

In [6]:
tf.random.normal(shape=[3, 4])

<tf.Tensor: shape=(3, 4), dtype=float32, numpy=
array([[-0.50018895,  1.1442305 , -0.13973145,  0.02657187],
       [-0.5047543 ,  1.4279577 , -0.215924  , -0.9652288 ],
       [-0.34063244, -0.90664583,  1.4439993 ,  0.10262676]],
      dtype=float32)>

Random numbers differ from one run to the next. For debugging, and for
results that others can reproduce, we fix the *seed* of the random number
generator, the value from which it computes its sequence of numbers:

TensorFlow keeps one global generator, which every draw advances, and
`tf.random.set_seed` resets it. Below, `a` and `c` are the first draws after
the same seed, so they agree, whereas `b` is a second draw and differs.

In [7]:
tf.random.set_seed(42)
a = tf.random.normal([3])
b = tf.random.normal([3])  # The generator has moved on
tf.random.set_seed(42)
c = tf.random.normal([3])  # Same seed: c repeats a
print(a, b, c, sep='\n')

tf.Tensor([ 0.3274685 -0.8426258  0.3194337], shape=(3,), dtype=float32)
tf.Tensor([ 0.08422458 -0.86090374  0.37812304], shape=(3,), dtype=float32)
tf.Tensor([ 0.3274685 -0.8426258  0.3194337], shape=(3,), dtype=float32)


Finally, we can write down the elements of a tensor ourselves, as nested
Python lists in which the outermost list runs along axis 0:

In [8]:
tf.constant([[2, 1, 4, 3], [1, 2, 3, 4], [4, 3, 2, 1]])

<tf.Tensor: shape=(3, 4), dtype=int32, numpy=
array([[2, 1, 4, 3],
       [1, 2, 3, 4],
       [4, 3, 2, 1]], dtype=int32)>

We can now build the running example. Until we load a real dataset, random
integers from 0 to 255 stand in for the pixels, because only the shape and
the type of the minibatch matter here:

In [9]:
# 256 grayscale images of 28 x 28 pixels, each pixel an integer in [0, 255];
# TensorFlow draws random integers as int32 or int64, so we cast them
images = tf.random.uniform((256, 1, 28, 28), maxval=256, dtype=tf.int32)
images = tf.cast(images, tf.uint8)
images.shape, images.dtype

(TensorShape([256, 1, 28, 28]), tf.uint8)

The shape reads: 256 images, 1 channel, 28 rows, and 28 columns.

## Reshaping and Indexing

### Reshaping

The same numbers often need a different arrangement. A classifier that
treats each image as a list of 784 pixel values needs the minibatch as a
matrix with one image per row, of shape $(256, 784)$. The function `reshape`
changes the shape of a tensor but neither its size nor the order of its
elements. On a small example, we turn the vector `x` of shape $(12,)$ into a
matrix `X` of shape $(3, 4)$. The elements fill the matrix one row at a
time, so `x[3]` becomes `X[0, 3]` (the figure):

![Reshaping a vector of 12 elements into a $3 \times 4$ matrix. The elements keep their order and fill the matrix one row at a time.](https://raw.githubusercontent.com/smolix/d2l-neu/notebooks/img/ndarray-reshape.svg)

In [10]:
X = tf.reshape(x, (3, 4))
X

<tf.Tensor: shape=(3, 4), dtype=float32, numpy=
array([[ 0.,  1.,  2.,  3.],
       [ 4.,  5.,  6.,  7.],
       [ 8.,  9., 10., 11.]], dtype=float32)>

Since the size is fixed, one entry of the target shape follows from the
others: 12 elements in 3 rows make 4 columns. Writing $-1$ for one entry
lets the library infer it, so `x.reshape(-1, 4)` and `x.reshape(3, -1)` give
the same matrix as `x.reshape(3, 4)`. In the same way, we flatten every
image of the minibatch into a row of 784 pixels without writing out 784:

In TensorFlow, `reshape` is a function rather than a method, as in
`tf.reshape(x, (-1, 4))`.

In [11]:
tf.reshape(images, (256, -1)).shape

TensorShape([256, 784])

A reshape need not move any data. The elements of `x` lie in memory one
after another, and the rows of `X` lie in memory one after another too, so a
library can implement the reshape by recording a new shape for the same
memory. Whether it does so matters only if elements can later be changed in
place, which some libraries allow.

TensorFlow tensors cannot be changed in place, so whether `tf.reshape`
copies the data cannot change any result.

Reshaping can also add or remove an axis of length 1, which leaves the
elements and their order unchanged. The first image of the minibatch,
`images[0]`, has shape $(1, 28, 28)$. A model that expects minibatches needs
it as a minibatch of one image, of shape $(1, 1, 28, 28)$, while a plot
needs a $28 \times 28$ matrix without the channel axis. The index `None`
inserts an axis of length 1, and `squeeze` removes one:

In TensorFlow, `tf.expand_dims(image, 0)` inserts the axis as well, and
`squeeze` is the function `tf.squeeze`.

In [12]:
image = images[0]
image[None].shape, tf.squeeze(image, 0).shape

(TensorShape([1, 1, 28, 28]), TensorShape([28, 28]))

### Indexing and Slicing

As with Python lists, we select elements of a tensor by *indexing*. Indices
start at 0, and negative indices count from the end. A *slice* `start:stop`
selects the positions from `start` up to, but not including, `stop`, and a
colon alone selects all positions. Indices for several axes are separated by
commas, and an index given alone applies to axis 0. Thus `X[-1]` selects the
last row of `X`, and `X[1:3]` selects the second and third rows:

In [13]:
X[-1], X[1:3]

(<tf.Tensor: shape=(4,), dtype=float32, numpy=array([ 8.,  9., 10., 11.], dtype=float32)>,
 <tf.Tensor: shape=(2, 4), dtype=float32, numpy=
 array([[ 4.,  5.,  6.,  7.],
        [ 8.,  9., 10., 11.]], dtype=float32)>)

The two results differ in shape. An integer index removes its axis, so
`X[-1]` is a vector, whereas a slice keeps its axis, so `X[1:3]` is a matrix
with two rows (the figure).

![Indexing a $3 \times 4$ matrix. The integer index `X[-1]` selects the last row and removes axis 0, leaving shape $(4,)$. The slice `X[1:3]` selects rows 1 and 2 and keeps axis 0, leaving shape $(2, 4)$.](https://raw.githubusercontent.com/smolix/d2l-neu/notebooks/img/ndarray-index-read.svg)

On the minibatch, `images[0]` is the first image, `images[:32]` a smaller
minibatch of the first 32 images, and `images[:, 0, 14, 14]` the pixel in row 14 and
column 14 of every image:

In [14]:
images[0].shape, images[:32].shape, images[:, 0, 14, 14].shape

(TensorShape([1, 28, 28]), TensorShape([32, 1, 28, 28]), TensorShape([256]))

An index can also be a tensor. Two tensors of integer positions, one for the
rows and one for the columns, select one element for each pair of
positions. This picks one entry from every row, as when we read off the
score that a classifier gives to the correct category of each image. The
matrix `X` holds the numbers 0 to 11 row by row, so the rows $(0, 1, 2)$ and
the columns $(3, 0, 2)$ read `X[0, 3]`, `X[1, 0]`, and `X[2, 2]`, which are
3, 4, and 10:

TensorFlow tensors do not accept tensors of positions as an index. The
function `tf.gather` with `batch_dims=1` picks, in every row of `X`, the
entry at the position that `cols` gives for that row:

In [15]:
cols = tf.constant([3, 0, 2])
tf.gather(X, cols, batch_dims=1)

<tf.Tensor: shape=(3,), dtype=float32, numpy=array([ 3.,  4., 10.], dtype=float32)>

A tensor of booleans with the shape of `X`, a *mask*, selects the elements
of `X` where it is `True`. The comparison `X > 2` compares every element of
`X` with 2 and gives such a mask, and the operator `&` combines two masks
elementwise into one that is `True` where both are (*and*); `|` gives *or*.
The selected elements need not form rows and columns, so the result is a
vector:

In [16]:
mask = (X > 2) & (X < 8)
X[mask]

<tf.Tensor: shape=(5,), dtype=float32, numpy=array([3., 4., 5., 6., 7.], dtype=float32)>

To keep the shape instead, `where(mask, A, B)` takes the elements of `A`
where the mask is `True` and those of `B` elsewhere. A single number in
place of `A` or `B` stands for that number at every position, so the
following sets every element outside the mask to 0:

In [17]:
tf.where(mask, X, 0)

<tf.Tensor: shape=(3, 4), dtype=float32, numpy=
array([[0., 0., 0., 3.],
       [4., 5., 6., 7.],
       [0., 0., 0., 0.]], dtype=float32)>

### Writing Elements

TensorFlow tensors are immutable: their elements cannot be assigned.
Mutable state, such as the parameters of a model, is held in a `Variable`,
whose elements the method `assign` overwrites. We copy `X` into a variable
and change one element:

In [18]:
X_var = tf.Variable(X)
X_var[1, 2].assign(17)
X_var

<tf.Variable 'Variable:0' shape=(3, 4) dtype=float32, numpy=
array([[ 0.,  1.,  2.,  3.],
       [ 4.,  5., 17.,  7.],
       [ 8.,  9., 10., 11.]], dtype=float32)>

To give many elements the same value, we index with a slice. Here
`[:2, :]` selects the first two rows, with the colon taking all columns:

In [19]:
X_var[:2, :].assign(12)
X_var

<tf.Variable 'Variable:0' shape=(3, 4) dtype=float32, numpy=
array([[12., 12., 12., 12.],
       [12., 12., 12., 12.],
       [ 8.,  9., 10., 11.]], dtype=float32)>

## Operations

### Elementwise Operations

An operation is *elementwise* if it applies a function of numbers to each
element of a tensor separately, or to each pair of elements at the same
position in two tensors of the same shape. Given a function $f$ of two
numbers, and two vectors $\mathbf{u}$ and $\mathbf{v}$ of the same length,
the elementwise operation returns the vector $\mathbf{c}$ with elements

$$c_i = f(u_i, v_i) \quad \textrm{for all } i.$$

One formula, applied at every position, thus turns a function of two numbers
into a function of two tensors of any shape. The arithmetic operators `+`,
`-`, `*`, `/`, and `**` (power) all work this way:

In [20]:
x = tf.constant([1.0, 2, 4, 8])
y = tf.constant([2.0, 2, 2, 2])
x + y, x - y, x * y, x / y, x ** y

(<tf.Tensor: shape=(4,), dtype=float32, numpy=array([ 3.,  4.,  6., 10.], dtype=float32)>,
 <tf.Tensor: shape=(4,), dtype=float32, numpy=array([-1.,  0.,  2.,  6.], dtype=float32)>,
 <tf.Tensor: shape=(4,), dtype=float32, numpy=array([ 2.,  4.,  8., 16.], dtype=float32)>,
 <tf.Tensor: shape=(4,), dtype=float32, numpy=array([0.5, 1. , 2. , 4. ], dtype=float32)>,
 <tf.Tensor: shape=(4,), dtype=float32, numpy=array([ 1.,  4., 16., 64.], dtype=float32)>)

Each result has the shape of the operands, and each element depends only on
the two elements at its position: `x / y` halves every element of `x`, and
`x ** y` squares it.

Functions of a single number, such as $e^x$, apply to each element in the
same way:

In [21]:
tf.exp(x)

<tf.Tensor: shape=(4,), dtype=float32, numpy=
array([2.7182817e+00, 7.3890562e+00, 5.4598148e+01, 2.9809580e+03],
      dtype=float32)>

The rescaling from the beginning of the section is elementwise as well.
Dividing the minibatch by 255 divides every pixel and returns a new tensor.
Division with `/` always returns floating-point numbers, here `float32`,
even where a quotient happens to be an integer. The second operand is a
single number rather than a tensor of the same shape, a case we return to
in the next subsection:

In [22]:
scaled = images / 255
print(scaled.dtype, images[0, 0, 0, :4], scaled[0, 0, 0, :4], sep='\n')

<dtype: 'float32'>
tf.Tensor([210 103 211 170], shape=(4,), dtype=uint8)
tf.Tensor([0.8235294  0.40392157 0.827451   0.6666667 ], shape=(4,), dtype=float32)


The type has changed from `uint8` to `float32`, and the first four pixels of
the first image, integers from 0 to 255, have become numbers between 0
and 1.

Comparisons, such as `X > 2` above, are elementwise too. They return a
tensor of booleans that is `True` where the comparison holds:

In [23]:
x == y

<tf.Tensor: shape=(4,), dtype=bool, numpy=array([False,  True, False, False])>

Not every operation is elementwise. Summing all the elements of a tensor
gives a tensor with a single element. Summing a tensor of booleans counts
its `True` entries, because `True` counts as 1 and `False` as 0; this is how
we will count the correct predictions of a classifier:

TensorFlow calls the sum `reduce_sum`, and booleans must first be converted
to numbers with `tf.cast`.

In [24]:
tf.reduce_sum(x), tf.reduce_sum(tf.cast(x == y, tf.int32))

(<tf.Tensor: shape=(), dtype=float32, numpy=15.0>,
 <tf.Tensor: shape=(), dtype=int32, numpy=1>)

A sum can also run along a single axis and leave, for instance, one sum per
row of a matrix. Such reductions along an axis are the subject of
that section.

### Broadcasting

Elementwise operations are simple when the two tensors have the same shape.
Model code, however, constantly combines a minibatch, stored with one example
per row, with a vector that holds one number per column. A layer adds the same
vector of offsets to its outputs for every example, and preprocessing subtracts
the same vector of column means from every example. Copying the vector once for
every example, only to match the shape of the minibatch, would waste memory and
time. *Broadcasting*, a convention that deep learning libraries adopted from
NumPy, applies the vector to every row without copying it. Here we add the
vector $(100, 200, 300, 400)$ to each row of a $3 \times 4$ matrix `X`:

In [25]:
X = tf.reshape(tf.range(12, dtype=tf.float32), (3, 4))
v = tf.constant([100.0, 200, 300, 400])
X + v

<tf.Tensor: shape=(3, 4), dtype=float32, numpy=
array([[100., 201., 302., 403.],
       [104., 205., 306., 407.],
       [108., 209., 310., 411.]], dtype=float32)>

Every row received the same four offsets, one per column.

Broadcasting follows a simple rule. Line the two shapes up from the *right*
and compare them axis by axis: two axes are compatible when they are equal
or when one of them is $1$ (a missing leading axis counts as $1$). Each axis
of length $1$ is then *stretched* to match the other, by virtually copying
its entries, and the elementwise operation is applied to the results. If any
pair of axes is incompatible, the operation raises an error rather than
guessing. For `X + v`, the shapes $(3, 4)$ and $(4,)$ line up as $(3, 4)$
and $(1, 4)$, so the single row of `v` is stretched to three rows. The
division of the minibatch by 255 was the simplest case: a scalar has no axes,
so it is stretched along all of them.

Both operands can be stretched at once. In
the figure, a column of shape $(3, 1)$ and a row of
shape $(1, 2)$ are each stretched to shape $(3, 2)$:

![Broadcasting stretches axes of length 1. The column `a` of shape $(3, 1)$ and the row `b` of shape $(1, 2)$ are both expanded to shape $(3, 2)$ (dashed: virtual copies) before the elementwise addition.](https://raw.githubusercontent.com/smolix/d2l-neu/notebooks/img/ndarray-broadcasting.svg)

In [26]:
a = tf.reshape(tf.range(3), (3, 1))
b = tf.reshape(tf.range(2), (1, 2))
a, b

(<tf.Tensor: shape=(3, 1), dtype=int32, numpy=
 array([[0],
        [1],
        [2]], dtype=int32)>,
 <tf.Tensor: shape=(1, 2), dtype=int32, numpy=array([[0, 1]], dtype=int32)>)

Their sum adds every element of the column to every element of the row:

In [27]:
a + b

<tf.Tensor: shape=(3, 2), dtype=int32, numpy=
array([[0, 1],
       [1, 2],
       [2, 3]], dtype=int32)>

Shapes that are not compatible raise an error. Lining up $(3, 2)$ and
$(2, 3)$ from the right pairs $2$ with $3$ and $3$ with $2$; in both pairs
the lengths differ and neither is $1$:

TensorFlow's error message also names the device and the internal operation
that failed, so we print only the type of the error.

In [28]:
try:
    tf.ones((3, 2)) + tf.ones((2, 3))
except Exception as e:
    print(type(e).__name__)

InvalidArgumentError


The rule also predicts what happens on the minibatch of images. Subtracting a
mean image of shape $(28, 28)$ from the minibatch of shape $(256, 1, 28, 28)$
works: the last two axes agree, and the mean image is stretched along the first
two. Adding a vector of 256 numbers, one per image, to the flattened minibatch
of shape $(256, 784)$ fails, because lining up the shapes from the right pairs
256 with 784. Reshaping the vector to shape $(256, 1)$ repairs this, since
$(256, 1)$ and $(256, 784)$ are compatible.

### Concatenation

Broadcasting combines tensors position by position. *Concatenation* instead
joins tensors end to end along an existing axis. Joining two minibatches of
256 images along axis 0 gives one minibatch of 512 images;
joining two sets of features for the same examples along axis 1 gives every
example more features. The lengths along all other axes must agree. We
concatenate two $3 \times 4$ matrices in both ways. Along axis 0, the result
has $3 + 3 = 6$ rows; along axis 1, it has $4 + 4 = 8$ columns:

In [29]:
X = tf.reshape(tf.range(12, dtype=tf.float32), (3, 4))
Y = tf.constant([[2.0, 1, 4, 3], [1, 2, 3, 4], [4, 3, 2, 1]])
tf.concat([X, Y], axis=0), tf.concat([X, Y], axis=1)

(<tf.Tensor: shape=(6, 4), dtype=float32, numpy=
 array([[ 0.,  1.,  2.,  3.],
        [ 4.,  5.,  6.,  7.],
        [ 8.,  9., 10., 11.],
        [ 2.,  1.,  4.,  3.],
        [ 1.,  2.,  3.,  4.],
        [ 4.,  3.,  2.,  1.]], dtype=float32)>,
 <tf.Tensor: shape=(3, 8), dtype=float32, numpy=
 array([[ 0.,  1.,  2.,  3.,  2.,  1.,  4.,  3.],
        [ 4.,  5.,  6.,  7.,  1.,  2.,  3.,  4.],
        [ 8.,  9., 10., 11.,  4.,  3.,  2.,  1.]], dtype=float32)>)

*Stacking* joins tensors of the same shape along a new axis instead. This is
how a minibatch arises from single examples: stacking 256 images of shape
$(1, 28, 28)$ gives the minibatch of shape $(256, 1, 28, 28)$. Stacking `X`
and `Y` along a new axis 0 gives shape $(2, 3, 4)$, and along a new last
axis, shape $(3, 4, 2)$:

In [30]:
tf.stack([X, Y]).shape, tf.stack([X, Y], axis=2).shape

(TensorShape([2, 3, 4]), TensorShape([3, 4, 2]))

## Memory and Conversion

### Allocation and In-Place Updates

Most operations return a new tensor, which needs new memory. The statement
`Y = Y + X` computes the sum in newly allocated memory and then binds the
name `Y` to the new tensor. Python's `id` function, which returns a number
that identifies an object for as long as the object exists, shows that `Y`
now names a different object:

In [31]:
before = id(Y)
Y = Y + X
id(Y) == before

False

For small tensors this costs little. A model with a billion parameters,
however, needs 4 GB to store them as `float32` numbers, and training updates
all of them at every step. If every update allocated new memory for
its result, the program would need room for the old and the new values at
the same time, and would spend time on the allocations. An *in-place*
update writes the result into the memory that the tensor already occupies.
It also reaches every part of the program that refers to the tensor,
whereas rebinding one name leaves other names pointing at the old values.
In-place updates have one restriction: a library that records operations to
compute derivatives may still need an old value, so an in-place update must
not overwrite it.

Ordinary TensorFlow tensors are immutable, so every operation on them
returns a new tensor. Model parameters are therefore stored in variables,
and `assign` writes a new value into a variable in place. (Inside a function
compiled with `tf.function`, TensorFlow can also reuse the memory of
intermediate results.) After the assignment, the variable `Z` is the same
object as before, and its elements hold `X + Y`:

In [32]:
Z = tf.Variable(tf.zeros_like(Y))
before = id(Z)
Z.assign(X + Y)
id(Z) == before

True

For ordinary tensors, `X += Y` is shorthand for `X = X + Y`, so it binds
`X` to a new tensor:

In [33]:
before = id(X)
X += Y
id(X) == before

False

### Conversion to Other Python Objects

Plotting, pandas, and many scientific libraries take NumPy arrays or Python
numbers rather than tensors, so we convert tensors at the boundary between
our code and theirs.

The method `numpy` copies a tensor into a NumPy array, and `tf.constant`
converts the array back, again by copying.

In [34]:
A = X.numpy()
B = tf.constant(A)
type(A), type(B)

(numpy.ndarray, tensorflow.python.framework.ops.EagerTensor)

Writing an element of the NumPy array `A` leaves `X` and `B` unchanged,
because each of them holds its own copy of the data:

In [35]:
A[0, 0] = 100  # Write to the NumPy copy
X[0, 0].numpy(), B[0, 0].numpy()

(np.float32(2.0), np.float32(2.0))

A tensor with a single element converts to a Python number with the method
`item` or with the built-in functions `float` and `int`. Printing a
summary number such as an average error, or comparing it with a threshold in
Python, needs such a conversion. When the
computation runs on a GPU, the conversion has to wait for it to finish, so
training code converts as rarely as possible.

TensorFlow tensors have no `item` method; the method `numpy` returns the
value as a NumPy number instead.

In [36]:
a = tf.constant(3.5)
a, a.numpy(), float(a), int(a)

(<tf.Tensor: shape=(), dtype=float32, numpy=3.5>, np.float32(3.5), 3.5, 3)

We can now create tensors, select and rearrange their elements, and compute
with whole tensors at once, including tensors whose shapes differ, and we
know when such computations allocate memory. The numbers in this section
were made up. Real data arrive in files, often as tables with missing
entries and with text in some of the columns; the next section turns such a
table into tensors.

## Summary

Deep learning code stores its data, parameters, and intermediate results in
tensors, arrays with a shape and a data type. It computes with whole tensors
rather than with loops over their elements: one expression divides all
200,704 pixels of a minibatch by 255. Reshaping, indexing, and
concatenation rearrange, select, and join elements. Elementwise operations
apply a function of numbers at every position, and broadcasting combines
tensors of different shapes by lining up their shapes from the right and
stretching axes of length 1. Most operations allocate a new tensor; where
memory matters, libraries with mutable tensors update them in place, and
libraries with immutable tensors reuse memory through mutable variables or
through their compiler.

## Exercises

1. [code] **Inferred reshape axis.** A minibatch of $n$ grayscale images of
   $28 \times 28$ pixels arrives as a tensor of shape $(n, 1, 28, 28)$.
   Each image must be flattened into a row of 784 values. Check every
   prediction below in code.
    1. Predict the result of reshaping the minibatch to $(-1, 784)$ and to
       $(256, -1)$, once for a full minibatch of $n = 256$ images and once
       for a final minibatch of $n = 100$ images. State which of the two
       targets works for every minibatch size.
    1. Suppose the images were loaded with three color channels by
       mistake, so that the minibatch has shape $(n, 3, 28, 28)$. Predict the
       result of the reshape to $(-1, 784)$, and state whether this step
       detects the mistake.

1. **Shifted slices.** A tensor `tokens` of shape $(4, 11)$ holds four token
   sequences, one per row. A model reads the first 10 tokens of each row
   and predicts, at every position, the token that follows. Write slice
   expressions for the input tensor and the target tensor, each of shape
   $(4, 10)$, so that the target at position $t$ is the token that
   follows the input at position $t$. Then give expressions for both
   tensors that stay correct without change when each row holds $T + 1$
   tokens and the model reads the first $T$, for every $T \geq 1$.

1. [code] **Broadcasting shapes.** Broadcasting aligns two shapes from the
   right.
    1. Replace `a` and `b` with two tensors that each have three axes,
       whose shapes differ on every axis and that still broadcast. Predict the shape of `a + b`,
       then run the code and check your prediction.
    1. A model returns predictions `y_hat` of shape $(5, 1)$ for five
       examples whose labels `y` form a vector of shape $(5,)$. Predict the
       shape of `y_hat - y`. Then let `y` hold the labels $0, 1, 2, 3, 4$
       as floating-point numbers and set `y_hat` to `y` reshaped to
       $(5, 1)$, so that every prediction equals its label. Compute the
       mean of the squared entries of `y_hat - y`, explain the value, and
       give the reshape of `y` that pairs each prediction with its own
       label.

1. [code] **Limits of data types.** The pixels of `images` are stored as
   `uint8`, and the rest of the book computes in `float32`.
    1. Predict the result of adding 100 to a `uint8` tensor that holds the
       value 200, and check your prediction. Explain the result, and
       brighten the first image of the minibatch by 100 intensity levels
       without this error by converting its type first.
    1. A `float32` number stores 24 significant binary digits. Explain why
       `float32` represents every integer up to $2^{24} = 16{,}777{,}216$,
       and check that adding 1 to $2^{24}$ in `float32` leaves it
       unchanged. A training loop counts the examples it has processed in a
       `float32` tensor. After how many examples does the count stop
       growing, and which data type should the counter have instead?

5. [code] **Variables and tensors as shared state.** Let `W = tf.Variable(X)`
   and `V = W`, so that two names refer to one variable, and let
   `T = tf.constant(X)` and `S = T`.
    1. Predict the value of `V` after `W.assign_add(Y)` and the value of
       `S` after `T += Y`, then check your predictions.
    1. Model parameters are referred to from several parts of a program
       and change at every training step. Explain from your results why
       TensorFlow stores them in variables rather than in tensors.

6. [code] **In-place composition.** ● Ordinary tensors are immutable, so
   every arithmetic operation on them allocates a new tensor. Hold `A` and
   `B` in `tf.Variable`s and compute `(A + B) * (-A / 2)` so that the
   result ends up in one of the two variables, using `assign`,
   `assign_add`, and related methods. List the steps of your solution,
   mark each step that still allocates a temporary tensor, and explain why
   eager execution cannot avoid these temporaries.

    *Adapted from Nicolas P. Rougier,
    [100 numpy exercises](https://github.com/rougier/numpy-100/blob/master/100_Numpy_exercises.md),
    problem 35.*

7. [code] **Loops against broadcasting.** ● For a vector `a` of 200
   elements and a vector `b` of 300 elements, compute the table of all
   absolute differences $d_{ij} = |a_i - b_j|$, of shape $(200, 300)$.
    1. Compute the table once with two nested Python loops, and once with
       a single broadcast expression after reshaping `a` to $(200, 1)$ and
       `b` to $(1, 300)$. Confirm that the two results agree, time both,
       and report the speedup.
    1. The broadcast expression copies neither input. How many bytes does
       its result take in `float32` when both vectors have 10,000
       elements? What does this say about the memory that broadcasting
       can create?

[Discussions](https://d2l.discourse.group/t/187)